In [6]:
import pandas as pd
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [7]:
def Read_data():
    os.getcwd()
    df_av = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS_AlgunaVez.csv')
    print(f'Cantidad de registros en el dataset de Alguna Vez: {len(df_av)}')
    df_av = df_av[df_av['Año'] <= 2020]
    print(f'Cantidad de registros en el dataset de Alguna Vez filtrado por año: {len(df_av)}')
    df_di = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS_DrogaImpacto.csv')
    df_um = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS_UltimoMes.csv')
    df_av_si = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS(SI)_AlgunaVez.csv')
    df_di_si = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS(SI)_DrogaImpacto.csv')
    df_um_si = pd.read_csv(f'{os.getcwd()}\\data\ECERIECS(SI)_UltimoMes.csv')
    return df_av, df_di, df_um, df_av_si, df_di_si, df_um_si

def Mod_sust(df):
    columnas_a_convertir = ['Tabaco', 'Alcohol', 'Marihuana', 'Hachis',	'Cocaina', 'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)',	'Solventes y removedores',	'Pegamento',	'Esmaltes y pinturas', 	'Otros (aire comprimido, gasolinas y combustibles)', 	'Anfetaminas',	'Metanfetaminas',	'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',	'Otros (derivados anfetaminicos)',	'LSD',	'Plantas alucinogenas y derivados',	'Otras (PCP, ketamina, excepto metanfetamina)',	'Benzodiazepinas',	'Rohypnol',	'Otras (sedantes hiptnoticos, GHB)',	'Heroina',	'Opiaceos sinteticos (propoxifeno, nailbufina)',	'Opio y opiodes (morfina, codeina)',	'Con utilidad medica (Prozac, Paxil, Carbamazepina)',	'Otras Sustancias']
    for c in columnas_a_convertir:
        df[c] = df[c].astype(str).str.lower().replace({"true": 1, "false": 0, "nan": None}) # Cambio map por replace

    for c in columnas_a_convertir:
        df[c] = df[c].astype('Int8') 
    
    return df

def Mod_Sociodemograficos(df):
    listas_columnas = ['ComunOcupacionId', 'SexoId', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunEstratoSocialId']
    for columna in listas_columnas:
        df[columna] = df[columna].fillna('Sin Dato')
        
    df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
                                                    'Sin ocupación': 2, 'Sin ocupacion': 2,
                                                    'Con actividad laboral': 3,
                                                    'Estudiante': 4,
                                                    'Hogar': 5,
                                                    'Sin Dato': 0}).astype('Int8')
    
    df['SexoId'] = df['SexoId'].replace({'Hombre': 1,
                                        'Mujer': 2,
                                        'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstadoCivilId'] = df['ComunEstadoCivilId'].replace({'Soltero(a)': 1,
                                                                'Casado(a)': 2,
                                                                'Divorciado(a)': 3,
                                                                'Unión libre': 4,
                                                                'Unión Libre': 4,
                                                                'Separado(a)': 5,
                                                                'Viudo(a)': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    
    df['ComunEscolaridadId'] = df['ComunEscolaridadId'].replace({'Sin Estudios': 1,
                                                                'Primaria': 2,
                                                                'Secundaria': 3,
                                                                'Preparatoria o Carrera Técnica': 4,
                                                                'Estudios Superiores': 5,
                                                                'Estudios de Posgrado': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstratoSocialId'] = df['ComunEstratoSocialId'].replace({'Muy Bajo': 1,
                                                                    'Bajo': 2,
                                                                    'Medio Bajo': 3,
                                                                    'Medio Alto': 4,                                                                   
                                                                    'Alto': 5,
                                                                    'Sin Dato': 0}).astype('Int8')
    return df

def DataClean(df):
    columnas_a_convertir3 = [
        "SexoId",
        "PerteneceComunidadLGBTTTI",
        "PerteneceComunidadIndigena",
        "PoblacionAfromexicanaAfroamericana",
        "DiscapacidadPerceptual",
        "Migracion"
    ]
    
    for columna in columnas_a_convertir3:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int8")

    df['Año'] = pd.to_numeric(df['Año'], errors="coerce").astype("Int16")
    # df['FolioId'] = pd.to_numeric(df['FolioId'], errors="coerce").astype("Int32")
    df['CentroCostoId'] = df['CentroCostoId'].fillna(99999).astype('Int32')
    columnas_eliminar = [
        "ConsumoDeDrogas",
        "ConsumoDeBebidasAlcoholicas",
        "ConsumoDeTabaco",
        "Ludopatia",
        "Otro",
        "TrastornosMentales",
        "Depresion",
        "Psicosis",
        "Epilepsia",
        "Demencia",
        "Autolesion",
        "Suicidio",
        "Ansiedad",
        "Mes", 
        'Edad_Años']
    
    columnas_eliminar2 = [col for col in df.columns if col.startswith('EdadInicio')]
    columnas_eliminar.extend(columnas_eliminar2)
    df.drop(columnas_eliminar, axis=1, inplace=True)
    df = df[df['Estado'].notna() & (df['Estado'] != "")]
    return df

def limpieza_datasets (df):
        #Se definen las columnas que serán filtradas del dataset
    filtro = ['Tabaco', 'Alcohol', 'Marihuana', 'Hachis', 'Cocaina', 'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)',	'Solventes y removedores',	'Pegamento',	'Esmaltes y pinturas', 	'Otros (aire comprimido, gasolinas y combustibles)', 	'Anfetaminas',	'Metanfetaminas',	'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',	'Otros (derivados anfetaminicos)',	'LSD',	'Plantas alucinogenas y derivados',	'Otras (PCP, ketamina, excepto metanfetamina)',	'Benzodiazepinas',	'Rohypnol',	'Otras (sedantes hiptnoticos, GHB)',	'Heroina',	'Opiaceos sinteticos (propoxifeno, nailbufina)',	'Opio y opiodes (morfina, codeina)',	'Con utilidad medica (Prozac, Paxil, Carbamazepina)',	'Otras Sustancias']
    # Verificar que las columnas existan
    columnas_existentes = [col for col in filtro if col in df.columns]
        # Filtrar filas en el mismo DataFrame (reasignando al índice filtrado)
    df.drop(df[~df[columnas_existentes].ge(1).any(axis=1)].index, inplace=True)
    return df

In [8]:
def main():
    df_av, df_di, df_um, df_av_si, df_di_si, df_um_si = Read_data()
    df_av = Mod_sust(df_av)
    df_di = Mod_sust(df_di)
    df_um = Mod_sust(df_um)
    df_av_si = Mod_sust(df_av_si)
    df_di_si = Mod_sust(df_di_si)
    df_um_si = Mod_sust(df_um_si)
    df_av = Mod_Sociodemograficos(df_av)
    df_di = Mod_Sociodemograficos(df_di)
    df_um = Mod_Sociodemograficos(df_um)
    df_av_si = Mod_Sociodemograficos(df_av_si)
    df_di_si = Mod_Sociodemograficos(df_di_si)
    df_um_si = Mod_Sociodemograficos(df_um_si)
    df_av = DataClean(df_av)
    df_di = DataClean(df_di)
    df_um = DataClean(df_um)
    df_av_si = DataClean(df_av_si)
    df_di_si = DataClean(df_di_si)
    df_um_si = DataClean(df_um_si)
    df_av = limpieza_datasets(df_av)
    df_di = limpieza_datasets(df_di)
    df_um = limpieza_datasets(df_um)
    df_av_si = limpieza_datasets(df_av_si)
    df_di_si = limpieza_datasets(df_di_si)
    df_um_si = limpieza_datasets(df_um_si)

    # df_av.to_csv(f'{os.getcwd()}\\results\\hist-SUST-AV-SLI.csv', index=False)
    # df_di.to_csv(f'{os.getcwd()}\\results\\hist-SUST-DI-SLI.csv', index=False)
    # df_um.to_csv(f'{os.getcwd()}\\results\\hist-SUST-UM-SLI.csv', index=False)
    # df_av_si.to_csv(f'{os.getcwd()}\\results\\hist-SUST-AV-SI.csv', index=False)
    # df_di_si.to_csv(f'{os.getcwd()}\\results\\hist-SUST-DI-SI.csv', index=False)
    # df_um_si.to_csv(f'{os.getcwd()}\\results\\hist-SUST-UM-SI.csv', index=False)
    return df_av, df_di, df_um, df_av_si, df_di_si, df_um_si

In [9]:
if __name__ == "__main__":
    df_av, df_di, df_um, df_av_si, df_di_si, df_um_si = main()

Cantidad de registros en el dataset de Alguna Vez: 442044
Cantidad de registros en el dataset de Alguna Vez filtrado por año: 442044


In [10]:
print(f'Tamaño del DataFrame df_av: {df_av.shape}')
print(f'Tamaño del DataFrame df_di: {df_di.shape}')
print(f'Tamaño del DataFrame df_um: {df_um.shape}')
print(f'Tamaño del DataFrame df_av_si: {df_av_si.shape}')
print(f'Tamaño del DataFrame df_di_si: {df_di_si.shape}')
print(f'Tamaño del DataFrame df_um_si: {df_um_si.shape}')

Tamaño del DataFrame df_av: (442018, 52)
Tamaño del DataFrame df_di: (418517, 54)
Tamaño del DataFrame df_um: (381303, 52)
Tamaño del DataFrame df_av_si: (363274, 52)
Tamaño del DataFrame df_di_si: (344920, 54)
Tamaño del DataFrame df_um_si: (316803, 52)
